# HGP on PEMS-BAY (Kaggle)

Horizon-specific causal graphs (Part A) and HICP (Part B) on PEMS-BAY, 15 / 30 / 60 minute horizons.
The notebook runs a list of settings (`SWEEP`). For each setting it builds the graphs once, trains every (configuration, seed)
pair in `RUNS`, and at the end prints a **validation** table to choose settings by.

**Current run:** `family` at hidden 128 (add `g12_all` and `none` in `RUNS` for the controls) with the physical and semantic graph branches switched on
(about 1 h on a T4). Each horizon branch mixes four terms through one softmax gate: the node itself, the causal graph G_h,
the road graph and a learned similarity graph. The printout after every run shows the mean gate weight of each branch.

**Before running**
1. Session options: Accelerator = GPU (T4 or P100). Internet = On if the code is cloned from GitHub (push the latest commit first).
2. Add as inputs: a PEMS-BAY dataset holding `pems-bay.h5` and `adj_mx_bay.pkl`.
3. Change only the settings cell. The training protocol (optimiser, learning rate, batch size, epochs, early stopping, split)
   is fixed in the code; only the method-specific settings are meant to be tuned.
4. Choose settings on validation error. Test metrics stay hidden until `SHOW_TEST = True` (final setting only).
5. Save a version ("Save & Run All") so `results_*` survives the session.

In [ ]:
# ---- settings
# The three controls of one setting, one seed. With PHYSICAL / SEMANTIC on, "none" = physical + semantic only (no causal graph).
# Final comparison: the same three for s in 42, 7, 123.
RUNS = [("family", 42)]      # family only for now; add ("g12_all", 42), ("none", 42) for the controls
HORIZONS = (3, 6, 12)      # tuning rounds. Final runs: (1, 3, 6, 12) adds 5 min; "mean" stays the mean over 15 / 30 / 60 min.
SHOW_TEST = False          # keep False while tuning; set True only for the final setting

# the base setting (already run once: results_base) and the method-specific hyperparameters
DEFAULTS = dict(
    FDR=0.3,           # knockoff+ level q: expected share of false edges among the selected ones
    GROUP_CORR=0.5,    # candidates with |correlation| above this form one knockoff group
    MAX_SOURCES=25,    # at most this many sources per target
    VAR_LAG=3,         # lags of the road-restricted VAR
    VAR_RIDGE=1e6,     # ridge penalty of the VAR (scales with the data: raw speeds here)
    HIDDEN=64,         # width of the temporal encoder
    DROPOUT=0.1,
    PHYSICAL=False,    # extra branch per horizon over the road adjacency (shared graph, own weights per horizon)
    SEMANTIC=False,    # extra branch per horizon over a learned similarity graph (node embeddings, top-25 sources per target)
)

# One entry per setting to try. "tag" names the folder (results_<tag>); the other keys override DEFAULTS.
# graphs_from="<tag>" reuses the graphs of an earlier entry (use it when only HIDDEN / DROPOUT change).
SWEEP = [
    # Extra graph branches at hidden 128. Compare with the causal-only runs of the same width (results_hidden128):
    # family 1.912, g12_all 1.913, none 2.005 (validation MAE). Graphs are built here (a minute or two), nothing is reused.
    dict(tag="h128_ps", HIDDEN=128, PHYSICAL=True, SEMANTIC=True),
]

REPO_URL = "https://github.com/prof-sanjay/HGP.git"
RUN_TESTS = True

## 1. Environment

In [ ]:
import glob
import os
import shutil
import subprocess
import sys

import torch

print(sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if not torch.cuda.is_available():
    print("warning: no GPU; PEMS-BAY (325 sensors) still runs on CPU but slowly")

## 2. Code and data

In [ ]:
WORK = "/kaggle/working"      # only results_<tag>/ (run folders and graphs/) is written here (the saved output)
SCRATCH = "/tmp/hgp"             # code, prepared data and logs: not part of the saved output
os.makedirs(SCRATCH, exist_ok=True)

hits = glob.glob("/kaggle/input/**/hgp/config.py", recursive=True)
if hits:
    CODE = os.path.dirname(os.path.dirname(hits[0]))
elif REPO_URL:
    CODE = f"{SCRATCH}/hgp_code"
    shutil.rmtree(CODE, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
    found = glob.glob(f"{CODE}/**/hgp/config.py", recursive=True)
    assert found, "the cloned repository has no hgp/config.py"
    CODE = os.path.dirname(os.path.dirname(found[0]))   # works if the repo root is HGP or contains an HGP/ folder
else:
    raise SystemExit("HGP code not found: add the repository as a Kaggle dataset or set REPO_URL")

files = [p for p in glob.glob("/kaggle/input/**/*", recursive=True) if os.path.isfile(p)]
h5_files = [p for p in files if p.lower().endswith(".h5")]
traffic = [p for p in h5_files if "pems" in os.path.basename(p).lower() and "meta" not in os.path.basename(p).lower()]
adjacency = [p for p in files if p.lower().endswith(".pkl") and "adj" in os.path.basename(p).lower()]
assert traffic, "no PEMS-BAY .h5 file under /kaggle/input (expected pems-bay.h5)"
assert adjacency, "no adjacency .pkl under /kaggle/input (expected adj_mx_bay.pkl)"
TRAFFIC, ADJACENCY = traffic[0], adjacency[0]
print("traffic  :", TRAFFIC)
print("adjacency:", ADJACENCY)

In [ ]:
def run(cmd, keep=None):
    """Run a command; print only the lines for which keep(line) is true (all lines if keep is None). Full log goes to a file."""
    name = os.path.basename(cmd[1]) if cmd[1] != "-m" else cmd[2]
    log = open(f"{SCRATCH}/log_{name}.txt", "a")
    with subprocess.Popen(cmd, cwd=CODE, env=dict(os.environ, PYTHONUNBUFFERED="1"), stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True) as proc:
        for line in proc.stdout:
            log.write(line)
            if keep is None or keep(line):
                print(line, end="")
    log.close()
    assert proc.returncode == 0, f"{name} exited with {proc.returncode}; see {SCRATCH}/log_{name}.txt"


def last_line(line):
    return " passed" in line or " failed" in line or "error" in line.lower()


def graph_lines(line):
    return line.startswith(("VAR fitted", "h=", "wrote", "Traceback")) and "targets (" not in line


def train_lines(line):
    s = line.strip()
    if s.startswith("epoch"):
        return int(s.split()[1]) % 10 == 0
    return SHOW_TEST and s.startswith("test")


if RUN_TESTS:
    run([sys.executable, "-m", "pytest", "tests/test_graphs.py", "tests/test_model.py", "tests/test_pems_bay.py", "-q",
         "-p", "no:cacheprovider", "-W", "ignore"], keep=last_line)

## 3. Prepare the data
Chronological 70 / 10 / 20 split, zeros are missing speeds, short gaps are filled for the model inputs only. The printout shows the size of the candidate pool: if it covers most of the network, knockoff selection has little to work with.

In [ ]:
DATA = f"{SCRATCH}/data/pems_bay.npz"
run([sys.executable, "scripts/prepare_data.py", "--dataset", "pems_bay", "--traffic", TRAFFIC, "--adjacency", ADJACENCY, "--out", DATA])

## 4. Part A: horizon-specific graphs (built once per setting, stored in results_<tag>/graphs)

In [ ]:
def setting(entry):
    s = dict(DEFAULTS)
    s.update({k: v for k, v in entry.items() if k not in ("tag", "graphs_from")})
    return s


def graph_dir(entry):
    return f"{WORK}/results_{entry.get('graphs_from', entry['tag'])}/graphs"


def build_graphs(entry):
    graphs, s = graph_dir(entry), setting(entry)
    if all(os.path.exists(f"{graphs}/G_h{h}.npz") for h in HORIZONS):
        print("graphs already built:", graphs)
        return
    assert "graphs_from" not in entry, f"graphs_from={entry['graphs_from']} has no graphs yet; list that entry earlier in SWEEP"
    run([sys.executable, "scripts/build_graphs.py", "--data", DATA, "--horizons", *map(str, HORIZONS), "--fdr", str(s["FDR"]),
         "--group-corr", str(s["GROUP_CORR"]), "--max-sources", str(s["MAX_SOURCES"]), "--var-lag", str(s["VAR_LAG"]),
         "--var-ridge", str(s["VAR_RIDGE"]), "--out", graphs], keep=graph_lines)

## 5. Part B: train and evaluate
Each run writes `results.json`, `history.csv`, `config.json`, `checkpoints/best.pt` and `test_predictions_h{3,6,12}.npz` into `results_<tag>/<run>/`.

In [ ]:
import glob
import json

for entry in SWEEP:
    tag, s = entry["tag"], setting(entry)
    print("=====", tag, {k: v for k, v in entry.items() if k != "tag"})
    build_graphs(entry)
    results = f"{WORK}/results_{tag}"
    for config, seed in RUNS:
        done = glob.glob(f"{results}/{config}_*seed{seed}/results.json") + glob.glob(f"{results}/{config}__seed{seed}/results.json")
        if done:
            print("already done:", config, seed, "->", done[0].split("/")[-2])
            continue
        print("---", config, "seed", seed)
        run([sys.executable, "scripts/train.py", "--config", config, "--seed", str(seed), "--data", DATA, "--horizons", *map(str, HORIZONS), "--graphs", graph_dir(entry),
             "--results", results, "--hidden", str(s["HIDDEN"]), "--dropout", str(s["DROPOUT"]),
             *(["--physical"] if s["PHYSICAL"] else []), *(["--semantic"] if s["SEMANTIC"] else [])], keep=train_lines)
        done = glob.glob(f"{results}/{config}_*seed{seed}/results.json") + glob.glob(f"{results}/{config}__seed{seed}/results.json")
        r = json.load(open(done[0]))
        gates = "  ".join(f"h{h}: " + "/".join(f"{v:.2f}" for v in g.values()) for h, g in r["test_gate_branches"].items())
        print(f"done: {r['run']}, best epoch {r['best_epoch']} of {r['epochs']} run (max 40), best val MAE {r['validation']['mean']['mae']:.3f}, "
              f"{r['training_seconds'] / 60:.1f} min\n      gate weights ({'/'.join(r['branches'])}): {gates}")

## 6. Validation table
Every run found under `/kaggle/working/results_*`, lowest validation MAE first. Choose settings from this table. Test metrics are shown only when `SHOW_TEST = True`.

In [ ]:
import glob
import json

rows = []
for f in glob.glob(f"{WORK}/results_*/*/results.json"):
    r = json.load(open(f))
    v = r["validation"]
    rows.append((v["mean"]["mae"], os.path.basename(os.path.dirname(os.path.dirname(f)))[len("results_"):], r["run"], v["3"]["mae"], v["6"]["mae"], v["12"]["mae"],
                 v["mean"]["rmse"], r["best_epoch"], r["training_seconds"] / 60, v["1"]["mae"] if "1" in v else float("nan")))
rows.sort()
print(f"{'tag':12s} {'run':30s} {'valMAE':>7s} {'15min':>6s} {'30min':>6s} {'60min':>6s} {'valRMSE':>8s} {'best':>4s} {'min':>5s} {'5min':>6s}")
for mae, tag, name, a, b, c, rmse, ep, minutes, m5 in rows:
    print(f"{tag:12s} {name:30s} {mae:7.3f} {a:6.3f} {b:6.3f} {c:6.3f} {rmse:8.3f} {ep:4d} {minutes:5.1f} {m5:6.3f}")

if SHOW_TEST:
    for d in sorted(glob.glob(f"{WORK}/results_*")):
        print("\n", d)
        run([sys.executable, "scripts/summarise.py", "--results", d, "--metric", "rmse"])